## CorrGAN
This notebook implements CorrGAN from [Marti, 2019](https://arxiv.org/abs/1910.09504) using stock correlations over the period 2001-2021.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
## PyTorch
import torch
import torch.nn as nn
import torch.utils.data as data
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.parametrizations import spectral_norm
# Torchvision
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid, save_image
import torchvision.transforms.functional as F
# tqdm
from tqdm.notebook import tqdm, trange
# statsmodel
from statsmodels.stats.correlation_tools import corr_nearest

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
def plot_correlation_heatmap(correlation_matrix, filename=""):
    fig, ax = plt.subplots()
    heatmap = ax.imshow(correlation_matrix, cmap='RdYlBu_r')
    cbar = plt.colorbar(heatmap)
    cbar.set_label('Correlation')

    # Set title and axis labels
    ax.set_title("Correlation Matrix Heatmap")

    # Show the plot
    plt.tight_layout()
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def show_tensor_images(image_tensor, 
                       num_images=64, 
                       size=(1, 32, 32), 
                       nrow=8, 
                       filename=""):
    image_unflat = image_tensor.detach().cpu().view(-1, *size)
    image_grid = make_grid(image_unflat[:num_images], nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### Load and Prepare the Dataset

In [ ]:
# Load the cleaned rolling correlation matrix from the CSV file
rolling_corr = pd.read_csv('rolling_corr.csv', index_col=[0,1], parse_dates=[0])

# Define the date
date = '2020-01-02'  # Change this to your desired date

# Extract the correlation matrix for the given date
corr_matrix = rolling_corr.loc[date]

# Convert the correlation matrix to a numpy array
corr_matrix_np = corr_matrix.values

In [ ]:
realcorr_filename = 'realcorr.png'
plot_correlation_heatmap(corr_matrix_np, filename=realcorr_filename)

In [ ]:
# Initialize a list to store the correlation matrices
corr_matrices = []

# Get the unique dates
dates = rolling_corr.index.get_level_values(0).unique()

# Loop through each date
for date in dates:
    # Extract the correlation matrix for the date
    corr_matrix = rolling_corr.loc[date].values
    # Append the correlation matrix to the list
    corr_matrices.append(corr_matrix)

# Convert the list of correlation matrices to a 3D PyTorch tensor
tensor_3d = torch.tensor(corr_matrices).float().to(device)

print(tensor_3d.shape)  # Should print: torch.Size([1, num_rows, num_columns])

In [ ]:
# Convert the 3D tensor to a TensorDataset
dataset = TensorDataset(tensor_3d)

# Convert the TensorDataset to a DataLoader
batch_size = 64
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

### Build the Generator class
The generator class represents the Generator network. The network has a noise vector as input and outputs correlation matrices of size (1, 32, 32). The network uses four transpose convolutions, three of which have a stride of 2 to expand the image size back up to 32 x 32. The first three layers use Batch Normalization and ReLU activation. The final layer does not have batch normalization and concludes with a Tanh function that matches the correlation scaling to the range [-1, 1].

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, out_size=32, no_channels=1):
        super(Generator, self).__init__()
        self.gen = nn.Sequential(
            nn.ConvTranspose2d( latent_dim, out_size * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(out_size * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(out_size * 4, out_size * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_size * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d( out_size * 2, out_size, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_size),
            nn.ReLU(True),
            nn.ConvTranspose2d( out_size, no_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )
        
    def forward(self, noise):
        return self.gen(noise)

### Build the Discriminator class
The discriminator class represents the Discriminator network that will be trained to determine if an image is generated or from the real data distribution. This is a standard classification network using three dense layers with Leaky ReLU activation. The discriminator uses spectral normalization.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, in_size=32, no_channels=1, neg_slope=0.2):
        super(Discriminator, self).__init__()
        self.disc = nn.Sequential(
            spectral_norm(nn.Conv2d(no_channels, in_size, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(in_size, in_size * 2, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(in_size * 2, in_size * 4, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(in_size * 4, 1, 4, 1, 0, bias=False)),
        )
        
    def forward(self, image):
        #print(image.shape)
        return self.disc(image)

### Train the Model
The loss functions must be defined first and then the training loop can be run.

In [ ]:
def gen_loss(gen, disc, batch_size, latent_dim, device):
    noise = torch.randn(batch_size, latent_dim, 1, 1, device=device)
    fake_im = gen(noise)
    fake_pred = disc(fake_im)
    gen_loss = -fake_pred.mean()
    return gen_loss

In [ ]:
def disc_loss(gen, disc, real_im, batch_size, latent_dim, device):
    noise = torch.randn(batch_size, latent_dim, 1, 1, device=device)
    fake_im = gen(noise).detach()
    fake_pred = disc(fake_im)
    fake_loss = fake_pred.mean()
    real_pred = disc(real_im)
    real_loss = -real_pred.mean()
    disc_loss = fake_loss + real_loss
    return disc_loss

In [ ]:
n_epochs = 50
latent_dim = 100
lr = 0.0001
beta_1 = 0.5
beta_2 = 0.9
epoch_show = 5
show_batch = 64
n_critic = 5

In [ ]:
gen = Generator(latent_dim=latent_dim).to(device)
gen_opt = Adam(gen.parameters(), lr=lr, betas=(beta_1, beta_2))
disc = Discriminator().to(device) 
disc_opt = Adam(disc.parameters(), lr=lr, betas=(beta_1, beta_2))

In [ ]:
gen_loss_lst = list()
disc_loss_lst = list()
tqdm_epoch = trange(n_epochs)
for epoch in tqdm_epoch:
    avg_gen_loss = 0.0
    avg_disc_loss = 0.0
    num_items = 0
    for [real_corr] in tqdm(dataloader):
        real_corr = real_corr.unsqueeze(1)
        for dsteps in range(n_critic):
            cur_batch_size = len(real_corr)

            # Update discriminator
            disc_opt.zero_grad()
        
            dloss = disc_loss(gen, disc, real_corr, cur_batch_size, 
                              latent_dim, device)
            dloss.backward(retain_graph=True)
            disc_opt.step()

        # Update Generator
        gen_opt.zero_grad()
        gloss = gen_loss(gen, disc, cur_batch_size, 
                         latent_dim, device)
        gloss.backward(retain_graph=True)
        gen_opt.step()

        avg_gen_loss += gloss.item() * cur_batch_size
        avg_disc_loss += dloss.item() * cur_batch_size
        num_items += cur_batch_size
        
    if epoch % epoch_show == 0:
        noise = torch.randn(show_batch, latent_dim, 1, 1, device=device)
        fake_corr = gen(noise)
        show_tensor_images((fake_corr + 1.0) / 2, filename='CorrGANGrid_' + str(epoch) + '.png')
        show_tensor_images((real_corr + 1.0) / 2)
        
    ave_gen_loss = avg_gen_loss / num_items
    ave_disc_loss = avg_disc_loss / num_items
    gen_loss_lst.append(ave_gen_loss)
    disc_loss_lst.append(ave_disc_loss)
    tqdm_epoch.set_description('Ave Gen Loss: {:5f}, \
                               Ave Disc Loss: {:5f}'.format(ave_gen_loss, ave_disc_loss))
    torch.save(gen.state_dict(), 'corrgan_genckpt.pth')
    torch.save(disc.state_dict(), 'corrgan_discckpt.pth')

### Plot Losses

In [ ]:
epochs = range(1, n_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
ax.plot(epochs, gen_loss_lst, label='Generator Loss')
ax.plot(epochs, disc_loss_lst, label='Discriminator Loss')
    
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('CorrGANLosses.png', dpi=300, bbox_inches='tight')

### Sample Correlation Matrix

In [ ]:
noise = torch.randn(1, latent_dim, 1, 1, device=device)
gen_corr = gen(noise)

In [ ]:
corrgan_filename = "CorrGANsample.png"
cgen_corr = gen_corr.squeeze().cpu().detach().numpy()
plot_correlation_heatmap(cgen_corr, corrgan_filename)

In [ ]:
np.linalg.cholesky(cgen_corr)

The matrix is likely not positive definite and needs to be regularized

In [ ]:
nearest_corr = corr_nearest(cgen_corr)

In [ ]:
np.linalg.cholesky(nearest_corr)

In [ ]:
corrgannear_filename = "CorrGANsample_nearest.png"
plot_correlation_heatmap(nearest_corr, corrgannear_filename)